In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path.cwd().parent / 'data' / 'interim'
ONBOARDING_WINDOW = 90   # observation window in days
BUFFER = 30              # extra days to confirm churn
DATA_END = pd.Timestamp('2032-01-14', tz='UTC')


In [ ]:
# Load NB01 outputs
new_clients  = pd.read_csv(DATA / 'new_clients.csv')
tx_with_reg  = pd.read_csv(DATA / 'tx_with_reg.csv')
new_clients['registration_date'] = pd.to_datetime(new_clients['registration_date'], utc=True)
tx_with_reg['date']              = pd.to_datetime(tx_with_reg['date'], utc=True)
tx_with_reg['registration_date'] = pd.to_datetime(tx_with_reg['registration_date'], utc=True)
print(f'new_clients: {len(new_clients):,} | tx_with_reg: {len(tx_with_reg):,}')


In [ ]:
# Days from registration to each transaction
tx_with_reg['days_since_reg'] = (
    (tx_with_reg['date'] - tx_with_reg['registration_date'])
    .dt.total_seconds() / 86400
)

# First transaction per client
first_tx = (
    tx_with_reg.sort_values('date')
    .groupby('client_id')['days_since_reg'].min()
    .reset_index()
    .rename(columns={'days_since_reg': 'days_since_first_tx'})
)
print(f'Clients with any transaction: {len(first_tx):,}')


In [ ]:
# Eligible: first tx in days 0..90, enough buffer before DATA_END
eligible_ids = first_tx[
    (first_tx['days_since_first_tx'] >= 0) &
    (first_tx['days_since_first_tx'] <= ONBOARDING_WINDOW)
]['client_id'].unique()

eligible = new_clients[new_clients['client_id'].isin(eligible_ids)].copy()
eligible = eligible.merge(first_tx, on='client_id', how='left')

# Drop clients whose buffer window extends past dataset end
eligible['cutoff'] = eligible['registration_date'] + pd.Timedelta(days=ONBOARDING_WINDOW + BUFFER)
eligible = eligible[eligible['cutoff'] <= DATA_END].copy()
print(f'Eligible clients: {len(eligible):,}')


In [ ]:
# CHURNED=1 if no transaction in days (90, 120] after first tx
active_after = tx_with_reg[
    tx_with_reg['client_id'].isin(eligible['client_id'])
].copy()
active_after = active_after.merge(
    eligible[['client_id', 'days_since_first_tx']], on='client_id', how='left'
)
active_after['days_from_first'] = active_after['days_since_reg'] - active_after['days_since_first_tx']
returners = active_after[
    (active_after['days_from_first'] > ONBOARDING_WINDOW) &
    (active_after['days_from_first'] <= ONBOARDING_WINDOW + BUFFER)
]['client_id'].unique()

eligible['churned'] = (~eligible['client_id'].isin(returners)).astype(int)
print(f'CHURNED  (1): {eligible["churned"].sum():,}  ({eligible["churned"].mean()*100:.1f}%)')
print(f'RETAINED (0): {(eligible["churned"]==0).sum():,}  ({(eligible["churned"]==0).mean()*100:.1f}%)')


In [ ]:
# Cohort stability: churn rate by registration month
eligible['reg_month'] = eligible['registration_date'].dt.to_period('M')
cohort = eligible.groupby('reg_month')['churned'].agg(['mean', 'count'])
cohort['mean'] = cohort['mean'] * 100
print(cohort[cohort['count'] > 500].to_string())


In [ ]:
# Save labelled dataset
eligible.to_csv(DATA / 'eligible_with_labels.csv', index=False)
print(f'Saved eligible_with_labels.csv ({len(eligible):,} rows)')
print(f'Churn rate: {eligible["churned"].mean()*100:.1f}%')
